# Preprocessing of RGI 08 (Norway) point mass balance data

This notebook builds the Norwegian stake (point mass balance, PMB) dataset used by MassBalanceMachine. The raw data are the NVE glaciological point mass balance dataset (https://doi.org/10.58059/sjse-6w92).

**Pipeline**
1. Fill missing start dates, split each row into a winter and an annual measurement and rename the columns to the MBM convention.
2. Download OGGM glacier directories and name glaciers from the RGI outlines; build unique stake IDs.
3. Fix inconsistent periods: January entered instead of October, manual fixes, and wrong `FROM_DATE` years.
4. Merge stakes closer than 10 m (same glacier, year and period).
5. Sample topographic variables from OGGM; drop stakes outside the RGI outline or without `aspect`, `slope` or `topo`.
6. Add the sky-view factor (SVF) from the per-glacier grids made by `1.4 Create_grids` (part 1).
7. Re-assign stake IDs by elevation.
8. Clean measurement months (drop Jul/Dec starts or ends, relabel some winter balances as annual).
9. Save `NOR_wgms_dataset_all.csv` to `cfg.dataPath/WGMS/Norway/csv/`.

## Setting up:

In [ ]:
import os, sys
sys.path.append(os.path.join(os.getcwd(), '../../../../')) # Add root of repo to import MBM

import warnings
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from tqdm import tqdm
from oggm import utils
import massbalancemachine as mbm

# Norway-specific helpers (fill_missing_dates, split_stake_measurements,
# check_period_consistency, fix_january_to_october_dates, flag_elevation_mismatch)
# and paths (path_PMB_WGMS_raw, path_PMB_WGMS_csv)
from papers.Transformer_TF.PMB_sources.RGI_08_Norway.scripts.norway_preprocess import *
from papers.Transformer_TF.PMB_sources.RGI_08_Norway.scripts.config_NOR import *

# Helper functions shared across regions (OGGM, stake QC, SVF, maps)
from papers.Transformer_TF.scripts.oggm import initialize_oggm_glacier_directories, export_oggm_grids, merge_pmb_with_oggm_data
from papers.Transformer_TF.scripts.pmb_processing import rename_stakes_by_elevation, check_point_ids_contain_glacier, remove_close_points
from papers.Transformer_TF.scripts.pmb_processing import add_svf_from_rgi_zarr, plot_missing_svf_for_all_glaciers, add_svf_nearest_valid
from papers.Transformer_TF.scripts.plotting.folium import plot_stakes_folium
warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

# Call config file (data paths, seed, etc.)
cfg = mbm.NorwayConfig()
mbm.utils.seed_all(cfg.seed)  # reproducibility
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

## Pre-processing:
Load the stakes, fill missing start dates, split into winter and annual, and convert to the WGMS/MBM format. The dataset comes from https://doi.org/10.58059/sjse-6w92.

### Process dates:
The raw file has one row per stake and year with three dates: `prev_yr_min_date` (end of the previous melt season), `curr_yr_max_date` (spring, maximum snow) and `curr_yr_min_date` (end of the current melt season). `split_stake_measurements` builds a winter row (`prev_yr_min_date` → `curr_yr_max_date`, `bw`) and an annual row (`prev_yr_min_date` → `curr_yr_min_date`, `ba`). Missing `prev_yr_min_date` values are first set to 1 October of the previous year.

In [ ]:
df_stakes = pd.read_csv(
    os.path.join(cfg.dataPath, path_PMB_WGMS_raw,
                 'glaciological_point_mass_balance_Norway.csv'))
# The NVE file already contains the RGI v6 ID of each stake
df_stakes = df_stakes.rename(columns={'rgiid': 'RGIId'})

# Add data modification column to keep track of mannual changes
df_stakes['DATA_MODIFICATION'] = ''

# FROM_DATE is missing in some glaciers despite having pmb measurements, fill with start of hydr. year
df_stakes = fill_missing_dates(df_stakes)

# Split into winter and annual measurements
df_stakes = split_stake_measurements(df_stakes)

# Transform to WGMS format
df_stakes = df_stakes.rename(
    columns={
        'lat': 'POINT_LAT',
        'lon': 'POINT_LON',
        'altitude': 'POINT_ELEVATION',
        'breid': 'GLACIER',
    })
# Only keep relevant columns in df
# (approx_loc / approx_altitude flag approximate positions; used in POINT_ID below)
df_stakes = df_stakes[[
    'POINT_LAT', 'POINT_LON', 'POINT_ELEVATION', 'FROM_DATE', 'TO_DATE',
    'POINT_BALANCE', 'PERIOD', 'RGIId', 'YEAR', 'GLACIER', 'DATA_MODIFICATION',
    'approx_loc', 'approx_altitude'
]]

# Convert dates from DD.MM.YYYY to YYYYMMDD
df_stakes['FROM_DATE'] = pd.to_datetime(
    df_stakes['FROM_DATE'], format='%d.%m.%Y').dt.strftime('%Y%m%d')
df_stakes['TO_DATE'] = pd.to_datetime(df_stakes['TO_DATE'],
                                      format='%d.%m.%Y').dt.strftime('%Y%m%d')

df_stakes.head()

### Add glacier names from the RGI ID
The OGGM glacier directories of the whole region are downloaded here. They are used below to name the glaciers and, later, to sample the topography.

In [ ]:
# initialize OGGM glacier directories (all glaciers of region 08)
gdirs, rgidf = initialize_oggm_glacier_directories(
    cfg,
    rgi_region="08",
    rgi_version="62",
    base_url=
    "https://cluster.klima.uni-bremen.de/~oggm/gdirs/oggm_v1.6/L1-L2_files/2025.6/elev_bands_w_data/",
    log_level='WARNING',
    task_list=None,
)

# Save the OGGM gridded data of these glaciers to disk
export_oggm_grids(cfg, gdirs, rgi_region="08")

In [ ]:
# Create a dictionary mapping from RGIId to glacier name
rgi_file = utils.get_rgi_region_file(region="08", version="62")
glacier_outline = gpd.read_file(rgi_file)

rgi_to_name_dict = dict(zip(glacier_outline.RGIId, glacier_outline.Name))
# Replaces the NVE glacier names ('breid') by the RGI names
df_stakes['GLACIER'] = df_stakes['RGIId'].map(rgi_to_name_dict)

# RGI60-08.02966 has no glacier name in the RGI map so directly give it name Blåbreen
df_stakes.loc[df_stakes['GLACIER'].isna(), 'GLACIER'] = 'Blabreen'

### Create unique POINT_ID:
The raw data have no stake ID, so a temporary ID is built from glacier, year, period, coordinates, the approximate-location flags and the row index. It is replaced by an elevation-based ID further down.

In [ ]:
# Create new POINT_ID column:
# <glacier>_<year>_<period>_<lat>_<lon>_<approx_loc>_<approx_altitude>_<row index>
df_stakes['POINT_ID'] = (df_stakes['GLACIER'] + '_' +
                         df_stakes['YEAR'].astype(str) + '_' +
                         df_stakes['PERIOD'].astype(str) + '_' +
                         df_stakes['POINT_LAT'].astype(str) + '_' +
                         df_stakes['POINT_LON'].astype(str) + '_' +
                         df_stakes['approx_loc'].astype(str) + '_' +
                         df_stakes['approx_altitude'].astype(str) + '_' +
                         df_stakes.index.astype(str))

# Drop columns that are not needed anymore
df_stakes = df_stakes.drop(columns=['approx_loc', 'approx_altitude'])

### Fix wrong date ranges:
`check_period_consistency` flags annual periods that do not last 9–15 months and winter periods that do not last 4–9 months.

In [ ]:
annual_inconsistent, winter_inconsistent = check_period_consistency(df_stakes)

# Show all inconsistent rows without truncation

pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
if len(annual_inconsistent) > 0:
    print("\nInconsistent annual periods:")
    display(annual_inconsistent[[
        'GLACIER', 'FROM_DATE', 'TO_DATE', 'MONTH_DIFF', 'PERIOD', 'YEAR',
        'RGIId', 'POINT_ID'
    ]])

if len(winter_inconsistent) > 0:
    print("\nInconsistent winter periods:")
    display(winter_inconsistent[[
        'GLACIER', 'FROM_DATE', 'TO_DATE', 'MONTH_DIFF', 'PERIOD', 'YEAR',
        'RGIId', 'POINT_ID'
    ]])

The first fix changes months entered as 01 (January) to 10 (October) in the flagged rows:

In [ ]:
# This function corrects the dates where 01 (Jan) has been entered as the month instead of 10 (Oct)
df_stakes_dates_fix = fix_january_to_october_dates(df_stakes,
                                                   annual_inconsistent,
                                                   winter_inconsistent)

annual_inconsistent, winter_inconsistent = check_period_consistency(
    df_stakes_dates_fix)

if len(annual_inconsistent) > 0:
    print("\nInconsistent annual periods:")
    display(annual_inconsistent[[
        'GLACIER', 'FROM_DATE', 'TO_DATE', 'MONTH_DIFF', 'PERIOD', 'YEAR',
        'RGIId', 'POINT_ID'
    ]])

if len(winter_inconsistent) > 0:
    print("\nInconsistent winter periods:")
    display(winter_inconsistent[[
        'GLACIER', 'FROM_DATE', 'TO_DATE', 'MONTH_DIFF', 'PERIOD', 'YEAR',
        'RGIId', 'POINT_ID'
    ]])

The second fix corrects some outliers by hand and removes stakes with nonsensical periods:

In [ ]:
## Fix outliers that don't have common explanation by hand
# May instead of september
df_stakes_dates_fix.loc[
    df_stakes_dates_fix['POINT_ID'] ==
    'Svartisheibreen_1994_annual_66.55012_13.72724_N_N_883',
    ['TO_DATE', 'DATA_MODIFICATION']] = [
        '19940915', 'Changed TO_DATE month from May to September'
    ]
df_stakes_dates_fix.loc[
    df_stakes_dates_fix['POINT_ID'] ==
    'Svartisheibreen_1994_annual_66.54826_13.73128_N_N_884',
    ['TO_DATE', 'DATA_MODIFICATION']] = [
        '19940915', 'Changed TO_DATE month from May to September'
    ]
# TO_DATE annual wrong year
df_stakes_dates_fix.loc[df_stakes_dates_fix['POINT_ID'] ==
                        'Aalfotbreen_1974_annual_61.74236_5.64623_N_N_1386',
                        ['TO_DATE', 'DATA_MODIFICATION']] = [
                            '19740920',
                            'Changed TO_DATE year from 1975 to 1974'
                        ]
df_stakes_dates_fix.loc[df_stakes_dates_fix['POINT_ID'] ==
                        'Aalfotbreen_1971_annual_61.75213_5.63165_N_N_1493',
                        ['TO_DATE', 'DATA_MODIFICATION']] = [
                            '19711124',
                            'Changed TO_DATE year from 1970 to 1971'
                        ]
df_stakes_dates_fix.loc[df_stakes_dates_fix['POINT_ID'] ==
                        'Graafjellsbrea_2009_annual_60.06923_6.38925_N_N_3545',
                        ['TO_DATE', 'DATA_MODIFICATION']] = [
                            '20091013',
                            'Changed TO_DATE year from 2019 to 2009'
                        ]
df_stakes_dates_fix.loc[df_stakes_dates_fix['POINT_ID'] ==
                        'Bondhusbrea_1981_annual_60.03108_6.31014_N_N_3738',
                        ['TO_DATE', 'DATA_MODIFICATION']] = [
                            '19810827',
                            'Changed TO_DATE year fomr 1980 to 1981'
                        ]
# TO_DATE winter wrong year
df_stakes_dates_fix.loc[
    df_stakes_dates_fix['POINT_ID'] ==
    'Langfjordjoekulen_2019_winter_70.12528_21.71827_N_N_4019',
    ['TO_DATE', 'DATA_MODIFICATION', 'YEAR', 'POINT_ID']] = [
        # NOTE: YEAR is set as a string here (and below), while the column is int
        '20200526', 'Changed TO_DATE year fomr 2019 to 2020', '2020',
        'Langfjordjoekulen_2020_winter_70.12528_21.71827_N_N_4019'
    ]

df_stakes_dates_fix.loc[
    df_stakes_dates_fix['POINT_ID'] ==
    'Blaaisen_1966_winter_68.33479_17.85005_N_N_4155',
    ['TO_DATE', 'DATA_MODIFICATION', 'YEAR', 'POINT_ID']] = [
        '19670520', 'Changed TO_DATE year fomr 1966 to 1967', '1967',
        'Blaaisen_1967_winter_68.33479_17.85005_N_N_4155'
    ]

df_stakes_dates_fix.loc[
    df_stakes_dates_fix['POINT_ID'] ==
    'Nigardsbreen_1963_winter_61.71461_7.11601_N_N_5802',
    ['TO_DATE', 'DATA_MODIFICATION', 'YEAR', 'POINT_ID']] = [
        '19640507', 'Changed TO_DATE year fomr 1963 to 1964', '1964',
        'Nigardsbreen_1964_winter_61.71461_7.11601_N_N_5802'
    ]

df_stakes_dates_fix.loc[
    df_stakes_dates_fix['POINT_ID'] ==
    'Vesledalsbreen_1967_winter_61.84804_7.25335_N_N_6694',
    ['TO_DATE', 'DATA_MODIFICATION', 'YEAR', 'POINT_ID']] = [
        '19680418', 'Changed TO_DATE year fomr 1967 to 1968', '1968',
        'Vesledalsbreen_1968_winter_61.84804_7.25335_N_N_6694'
    ]

df_stakes_dates_fix.loc[
    df_stakes_dates_fix['POINT_ID'] ==
    'Hellstugubreen_2010_winter_61.57329_8.44438_N_N_6935',
    ['TO_DATE', 'DATA_MODIFICATION', 'YEAR', 'POINT_ID']] = [
        '20110505', 'Changed TO_DATE year fomr 2010 to 2011', '2011',
        'Hellstugubreen_2011_winter_61.57329_8.44438_N_N_6935'
    ]

# These stakes have nonsensical periods, remove them out of df and index list
stakes_to_remove = [
    'Austdalsbreen_2017_annual_61.81113_7.36766_Y_N_3038',
    'Austdalsbreen_2017_annual_61.80888_7.38239_Y_N_3065',
    'Aalfotbreen_1967_winter_61.74294_5.6365_N_N_5379',
    'Hansebreen_2012_winter_61.74307_5.66278_N_N_5625',
    'Austdalsbreen_2017_winter_61.81113_7.36766_Y_N_6792',
    'Austdalsbreen_2017_winter_61.80888_7.38239_Y_N_6819'
]
df_stakes_dates_fix = df_stakes_dates_fix[~df_stakes_dates_fix['POINT_ID'].
                                          isin(stakes_to_remove)]

# Check again: the remaining inconsistencies are handled in the next cell
annual_inconsistent, winter_inconsistent = check_period_consistency(
    df_stakes_dates_fix)

if len(annual_inconsistent) > 0:
    print("\nInconsistent annual periods:")
    display(annual_inconsistent[[
        'GLACIER', 'FROM_DATE', 'TO_DATE', 'MONTH_DIFF', 'PERIOD', 'YEAR',
        'RGIId', 'POINT_ID'
    ]])

if len(winter_inconsistent) > 0:
    print("\nInconsistent winter periods:")
    display(winter_inconsistent[[
        'GLACIER', 'FROM_DATE', 'TO_DATE', 'MONTH_DIFF', 'PERIOD', 'YEAR',
        'RGIId', 'POINT_ID'
    ]])
pd.reset_option('display.max_rows')
pd.reset_option('display.max_colwidth')

The remaining inconsistencies all have a wrong `FROM_DATE` year, which is set to the year before `YEAR`:

In [ ]:
remaining_indices = list(annual_inconsistent.index) + list(
    winter_inconsistent.index)

# For each remaining inconsistent record, change the year in FROM_DATE to the previous year
for idx in remaining_indices:
    # Get year from the YEAR column
    year = int(df_stakes_dates_fix.loc[idx, 'YEAR']) - 1

    # Extract month and day part from current FROM_DATE (keeping positions 4-8 which contain MMDD)
    month_day = df_stakes_dates_fix.loc[idx, 'FROM_DATE'][4:8]

    # Create new FROM_DATE by combining YEAR with the extracted month_day
    df_stakes_dates_fix.loc[idx, 'FROM_DATE'] = f"{year}{month_day}"
    df_stakes_dates_fix.loc[
        idx,
        'DATA_MODIFICATION'] = "Changed faulty year in FROM_DATE to previous year of TO_DATE"

# Should now report 0 inconsistent periods
annual_inconsistent, winter_inconsistent = check_period_consistency(
    df_stakes_dates_fix)

# Display the inconsistent records
if len(annual_inconsistent) > 0:
    print("\nInconsistent annual periods:")
    display(annual_inconsistent[[
        'GLACIER', 'FROM_DATE', 'TO_DATE', 'MONTH_DIFF', 'PERIOD', 'YEAR',
        'RGIId', 'POINT_ID'
    ]])

if len(winter_inconsistent) > 0:
    print("\nInconsistent winter periods:")
    display(winter_inconsistent[[
        'GLACIER', 'FROM_DATE', 'TO_DATE', 'MONTH_DIFF', 'PERIOD', 'YEAR',
        'RGIId', 'POINT_ID'
    ]])

### Merge close stakes:
Within each glacier, year and period, stakes closer than 10 m are merged into one point with the mean `POINT_BALANCE` (see `remove_close_points`). The data contain only annual and winter rows (see `split_stake_measurements`).

In [ ]:
# Merge stakes < 10 m apart, glacier by glacier
df_stakes_merged = pd.DataFrame()
for gl in tqdm(df_stakes_dates_fix.GLACIER.unique(), desc='Merging stakes'):
    print(f'-- {gl.capitalize()}:')
    df_gl = df_stakes_dates_fix[df_stakes_dates_fix.GLACIER == gl]
    df_gl_cleaned = remove_close_points(df_gl)
    df_stakes_merged = pd.concat([df_stakes_merged, df_gl_cleaned])
# Drop temporary projected coordinates added by remove_close_points
df_stakes_merged.drop(['x', 'y'], axis=1, inplace=True)
df_stakes_merged.reset_index(inplace=True, drop=True)

### Add OGGM data:
Sample the topographic variables at the nearest OGGM grid cell of each stake and flag whether the stake lies within its RGI outline (`within_glacier_shape`).

In [ ]:
# Sample topographic variables at each stake and flag stakes within the glacier outline
# (default variables: aspect, slope, topo, hugonnet_dhdt, consensus_ice_thickness, millan_v)
df_pmb_topo = merge_pmb_with_oggm_data(df_pmb=df_stakes_merged,
                                       gdirs=gdirs,
                                       rgi_region="08",
                                       rgi_version="62")

Only keep stakes within the RGI outline, and drop stakes where `aspect`, `slope` or `topo` is NaN.

In [ ]:
# --- Initial size ---
n_start = len(df_pmb_topo)

# Restrict to within glacier shape
mask_within = df_pmb_topo['within_glacier_shape'] == True
n_drop_shape = (~mask_within).sum()
df_pmb_topo = df_pmb_topo.loc[mask_within].copy()
df_pmb_topo = df_pmb_topo.drop(columns=['within_glacier_shape'])

print(f"Dropped {n_drop_shape} points outside glacier shape")

# Drop rows with NaN in the topographic model inputs (aspect, slope, topo)
mask_nan_topo = df_pmb_topo[['aspect', 'slope', 'topo']].isna().any(axis=1)
n_drop_topo = mask_nan_topo.sum()
df_pmb_topo = df_pmb_topo.loc[~mask_nan_topo].copy()

print(f"Dropped {n_drop_topo} points with NaN aspect, slope or topo")

# --- Final counts ---
print(f"Total dropped: {n_start - len(df_pmb_topo)}")
print('Number of winter and annual samples:', len(df_pmb_topo))
print('Number of annual samples:',
      len(df_pmb_topo[df_pmb_topo.PERIOD == 'annual']))
print('Number of winter samples:',
      len(df_pmb_topo[df_pmb_topo.PERIOD == 'winter']))

# Unique glaciers, sorted
glacier_list = sorted(df_pmb_topo.GLACIER.unique())
print(f"Number of glaciers: {len(glacier_list)}")
print(f"Glaciers: {glacier_list}")

In [ ]:
# Number of measurements per year (top) and per glacier (bottom)
# Number of measurements per year:
fig, axs = plt.subplots(2, 1, figsize=(20, 15))
ax = axs.flatten()[0]
df_pmb_topo.groupby(['YEAR', 'PERIOD']).size().unstack().plot(
    kind='bar',
    stacked=True,
    color=[mbm.plots.COLOR_ANNUAL, mbm.plots.COLOR_WINTER],
    ax=ax)
ax.set_title('Number of measurements per year for all glaciers')

ax = axs.flatten()[1]
num_gl = df_pmb_topo.groupby(['GLACIER']).size().sort_values()
num_gl.plot(kind='bar', ax=ax)
ax.set_title('Number of total measurements per glacier since 1951')
plt.tight_layout()

### Check for wrong elevation:
Lists stakes whose reported elevation differs from the OGGM DEM (`topo`) by more than 400 m. This is a diagnostic only: the flagged rows are not removed.

In [ ]:
# Diagnostic only: df_checked / df_bad are not used further
df_checked, df_bad = flag_elevation_mismatch(df_pmb_topo, threshold=400)

### Add sky-view factor:
The SVF is sampled at each stake from the per-glacier grids (`<RGIId>.zarr` in `xr_masked_grids/`). Stakes on a masked (NaN) pixel are first plotted, then filled with the nearest valid SVF pixel.

In [ ]:
# Directory with the per-glacier SVF grids (<RGIId>.zarr)
path_masked_xr = os.path.join(cfg.dataPath,
                              "RGI_v6/RGI_08_Scandinavia/xr_masked_grids")

# Nearest-pixel SVF sampling (NaN if the pixel is masked)
df_pmb_topo_svf = add_svf_from_rgi_zarr(
    df_pmb_topo,
    path_masked_xr,
    rgi_col="RGIId",
    lon_col="POINT_LON",
    lat_col="POINT_LAT",
    svf_var="svf",
    out_col="svf",
)
# Stakes that fell on a masked (NaN) SVF pixel
df_missing = df_pmb_topo_svf[df_pmb_topo_svf["svf"].isna()].copy()
print("Missing SVF points:", len(df_missing))
print("Glaciers affected:", sorted(df_missing["RGIId"].unique()))

In [ ]:
# Plot SVF grids with missing-SVF stakes (red) and valid stakes (white) per glacier
plot_missing_svf_for_all_glaciers(
    df_with_svf=df_pmb_topo_svf,
    path_masked_xr=path_masked_xr,
    plot_valid_points=True,
    save_dir=
    None  # or e.g. os.path.join(cfg.dataPath, "diagnostics/svf_missing")
)

In [ ]:
# Sample SVF again; points on masked pixels are filled with the nearest
# valid pixel within max_radius grid cells. This is the SVF that is kept.
df_pmb_topo_svf_new = add_svf_nearest_valid(
    df_pmb_topo,
    path_masked_xr,
    rgi_col="RGIId",
    lon_col="POINT_LON",
    lat_col="POINT_LAT",
    svf_var="svf",
    out_col="svf",
    max_radius=30,  # ~30 grid cells search; adjust if needed
)

print("Missing SVF points after nearest-valid fill:",
      df_pmb_topo_svf_new["svf"].isna().sum())

# Same plot after filling: there should be no red points left
plot_missing_svf_for_all_glaciers(
    df_with_svf=df_pmb_topo_svf_new,
    path_masked_xr=path_masked_xr,
    plot_valid_points=True,
    save_dir=
    None  # or e.g. os.path.join(cfg.dataPath, "diagnostics/svf_missing")
)

### Give new stake IDs:
Stake IDs are re-assigned per glacier as `<GLACIER>_<rank>`, where the rank is the stake's mean elevation (0 = lowest).

In [ ]:
# Rename POINT_IDs to <GLACIER>_<rank>, ranked by elevation
df_pmb_new_ids = rename_stakes_by_elevation(df_pmb_topo_svf_new)

# Check that every POINT_ID contains its glacier name
check_point_ids_contain_glacier(df_pmb_new_ids)

print('Number of winter and annual samples:', len(df_pmb_new_ids))
print('Number of annual samples:',
      len(df_pmb_new_ids[df_pmb_new_ids.PERIOD == 'annual']))
print('Number of winter samples:',
      len(df_pmb_new_ids[df_pmb_new_ids.PERIOD == 'winter']))

# Histogram of mass balance
df_pmb_new_ids['POINT_BALANCE'].hist(bins=20)
plt.xlabel('Mass balance [m w.e.]')

## Final cleaning:
1. Remove measurements that start or end in July or December.
2. Relabel "winter" measurements that end in June with a negative balance as annual.

In [ ]:
df_pmb_clean = df_pmb_new_ids.copy()

# Ensure YYYYMMDD format
df_pmb_clean["FROM_DATE"] = df_pmb_clean["FROM_DATE"].astype(str).str.zfill(8)
df_pmb_clean["TO_DATE"] = df_pmb_clean["TO_DATE"].astype(str).str.zfill(8)

# Extract months (MM) of start and end of each measurement
df_pmb_clean["MONTH_START"] = df_pmb_clean["FROM_DATE"].str[4:6]
df_pmb_clean["MONTH_END"] = df_pmb_clean["TO_DATE"].str[4:6]


# Helper: print unique start/end months for winter and annual measurements
def print_months(df, label):
    winter = df[df.PERIOD == "winter"]
    annual = df[df.PERIOD == "annual"]

    print(f"\n{label}")
    print("Winter measurement months:")
    print("  Unique start months:", sorted(winter["MONTH_START"].unique()))
    print("  Unique end months:  ", sorted(winter["MONTH_END"].unique()))

    print("\nAnnual measurement months:")
    print("  Unique start months:", sorted(annual["MONTH_START"].unique()))
    print("  Unique end months:  ", sorted(annual["MONTH_END"].unique()))


# --- Before filtering ---
print_months(df_pmb_clean, "Before filtering")

# --- Remove unwanted months in start/end (July + December) ---
# These do not fit the hydrological year (Oct-Sep) used by MBM
bad_months = {"07", "12"}
mask_bad_months = (df_pmb_clean["MONTH_START"].isin(bad_months)
                   | df_pmb_clean["MONTH_END"].isin(bad_months))
n_removed = mask_bad_months.sum()

df_pmb_clean = df_pmb_clean.loc[~mask_bad_months].copy()
print(
    f"\nRemoved {n_removed} rows with MONTH_START or MONTH_END in {sorted(bad_months)}."
)

# --- Correct mislabeled winter MB ---
# If MONTH_END == 06 and MB is negative, it should be annual (not winter):
# the balance already includes melt
mask_fix = ((df_pmb_clean["PERIOD"].str.strip().str.lower() == "winter") &
            (df_pmb_clean["MONTH_END"] == "06") &
            (df_pmb_clean["POINT_BALANCE"] < 0))
print("Rows to relabel winter -> annual:", int(mask_fix.sum()))

df_pmb_clean.loc[mask_fix, "PERIOD"] = "annual"

# --- After filtering + relabeling ---
print_months(df_pmb_clean, "After filtering + relabeling")

## Save final dataframe:

In [ ]:
# Save to csv:
df_pmb_clean.to_csv(os.path.join(cfg.dataPath, path_PMB_WGMS_csv,
                                 'NOR_wgms_dataset_all.csv'),
                    index=False)

# Histogram of mass balance
df_pmb_clean['POINT_BALANCE'].hist(bins=20)
plt.xlabel('Mass balance [m w.e.]')

## Plot stakes:

In [ ]:
# Interactive map of all cleaned stakes, one layer per glacier

m = plot_stakes_folium(df_pmb_clean, color_map=None, layer_control=True)
m